# STL-10 Image Classification with MLOps on Google Colab

Complete STL-10 image classification using ResNet-18 with MLOps pipeline including Weights & Biases logging.

**Features:**
- ✅ Load STL-10 data from HuggingFace
- ✅ ResNet-18 pretrained model training
- ✅ Wandb experiment tracking
- ✅ Confusion matrix and class-wise accuracy
- ✅ Sample predictions visualization
- ✅ GPU acceleration
- ✅ Automatic exam answers generation

**Author:** Shivam Madhav Kenche (M25CSA028)

## 🚀 Setup and Installation

In [ ]:
# Install required packages
!pip install torch torchvision transformers datasets wandb huggingface_hub matplotlib seaborn scikit-learn tqdm
!pip install --upgrade wandb

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from datasets import load_dataset
from PIL import Image
import wandb
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

print("📦 All packages imported successfully!")

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available, using CPU (training will be slower)")

## ⚙️ Configuration

In [ ]:
# Configuration class
class Config:
    # Dataset configuration
    DATASET_NAME = "Chiranjeev007/STL-10_Subset"
    NUM_CLASSES = 10
    CLASS_NAMES = [
        "airplane", "bird", "car", "cat", "deer", 
        "dog", "horse", "monkey", "ship", "truck"
    ]
    
    # Model configuration
    MODEL_NAME = "resnet18"
    PRETRAINED = True
    
    # Training configuration
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    NUM_EPOCHS = 15  # Reduced for Colab
    PATIENCE = 3     # Reduced for faster training
    
    # Data augmentation
    IMG_SIZE = 224
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    
    # Wandb configuration
    WANDB_PROJECT = "stl10-classification-colab"
    WANDB_ENTITY = None
    
    # Paths (Colab friendly)
    MODEL_SAVE_PATH = "/content/models"
    RESULTS_PATH = "/content/results"
    
    # Device
    DEVICE = device

config = Config()
print("✅ Configuration set up successfully!")
print(f"📊 Project: {config.WANDB_PROJECT}")
print(f"🎯 Device: {config.DEVICE}")
print(f"📦 Batch size: {config.BATCH_SIZE}")
print(f"🔄 Max epochs: {config.NUM_EPOCHS}")

## 📊 Weights & Biases Setup

In [ ]:
# Login to Wandb
# Option 1: Use the provided API key
wandb_api_key = "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"
os.environ["WANDB_API_KEY"] = wandb_api_key

# Option 2: Login interactively (uncomment if needed)
# wandb.login()

# Initialize wandb
wandb.init(
    project=config.WANDB_PROJECT,
    entity=config.WANDB_ENTITY,
    config={
        "batch_size": config.BATCH_SIZE,
        "learning_rate": config.LEARNING_RATE,
        "epochs": config.NUM_EPOCHS,
        "model": config.MODEL_NAME,
        "dataset": config.DATASET_NAME,
        "device": str(config.DEVICE)
    }
)

print("✅ Wandb initialized successfully!")
print(f"🔗 Dashboard: {wandb.run.url}")

## 📁 Data Loading and Preprocessing

In [ ]:
# Custom Dataset class
class STL10Dataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        label = item['label']
        
        # Convert to PIL Image if needed
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

print("✅ Dataset class defined!")

In [ ]:
# Data transformations
train_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

val_test_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

print("✅ Transforms defined!")
print(f"📊 Train transforms: {len(train_transform.transforms)} operations")
print(f"🔍 Test transforms: {len(val_test_transform.transforms)} operations")

In [ ]:
# Load dataset from HuggingFace
print(f"📥 Loading dataset: {config.DATASET_NAME}")
dataset = load_dataset(config.DATASET_NAME)

# Create datasets
train_dataset = STL10Dataset(dataset['train'], transform=train_transform)
val_dataset = STL10Dataset(dataset['validation'], transform=val_test_transform)
test_dataset = STL10Dataset(dataset['test'], transform=val_test_transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=2,  # Reduced for Colab
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True
)

print(f"✅ Data loaded successfully!")
print(f"🚂 Train samples: {len(train_dataset)}")
print(f"🔍 Validation samples: {len(val_dataset)}")
print(f"🧪 Test samples: {len(test_dataset)}")
print(f"📦 Classes: {config.CLASS_NAMES}")

## 🧠 Model Definition

In [ ]:
# Create ResNet-18 model
def create_model():
    """Create ResNet-18 model for STL-10 classification"""
    model = models.resnet18(pretrained=config.PRETRAINED)
    
    # Modify final layer for STL-10 (10 classes)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, config.NUM_CLASSES)
    
    return model

# Create model and move to device
model = create_model().to(config.DEVICE)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print("✅ Model created successfully!")
print(f"🧠 Model: ResNet-18")
print(f"📊 Output classes: {config.NUM_CLASSES}")
print(f"🎯 Device: {next(model.parameters()).device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📈 Total parameters: {total_params:,}")
print(f"🔧 Trainable parameters: {trainable_params:,}")

## 🏃‍♂️ Training Pipeline

In [ ]:
# Create directories
os.makedirs(config.MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(config.RESULTS_PATH, exist_ok=True)
print(f"📁 Created directories: {config.MODEL_SAVE_PATH}, {config.RESULTS_PATH}")

In [ ]:
# Training functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()
        
        current_accuracy = correct_predictions / total_samples
        pbar.set_postfix({
            'loss': f"{running_loss / (pbar.n + 1):.4f}",
            'acc': f"{current_accuracy:.4f}"
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct_predictions / total_samples
    
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct_predictions / total_samples
    
    return epoch_loss, epoch_acc

print("✅ Training functions defined!")

In [ ]:
# Training loop
print("🚀 Starting training...")
print(f"📊 Total epochs: {config.NUM_EPOCHS}")
print(f"⏰ Patience: {config.PATIENCE}")
print("="*60)

best_val_acc = 0.0
patience_counter = 0

for epoch in range(config.NUM_EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{config.NUM_EPOCHS}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, config.DEVICE)
    
    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, config.DEVICE)
    
    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
    
    print(f"📈 Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"📊 Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        
        # Save checkpoint
        best_model_path = os.path.join(config.MODEL_SAVE_PATH, "best_model.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'train_accuracy': train_acc,
        }, best_model_path)
        
        print(f"💾 New best model saved! Validation accuracy: {val_acc:.4f}")
    else:
        patience_counter += 1
        print(f"⏰ Patience: {patience_counter}/{config.PATIENCE}")
        
    # Early stopping
    if patience_counter >= config.PATIENCE:
        print(f"🛑 Early stopping triggered after {epoch+1} epochs")
        break

print(f"\n🎉 Training completed!")
print(f"🏆 Best validation accuracy: {best_val_acc:.4f}")
print("="*60)

## 📊 Evaluation and Visualization

In [ ]:
# Load best model for evaluation
print("📥 Loading best model for evaluation...")
best_model_path = os.path.join(config.MODEL_SAVE_PATH, "best_model.pth")
checkpoint = torch.load(best_model_path, map_location=config.DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Best model loaded (Val Acc: {checkpoint['val_accuracy']:.4f})")

In [ ]:
# Evaluate on test set
print("🧪 Evaluating on test set...")

model.eval()
correct_predictions = 0
total_samples = 0
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = correct_predictions / total_samples
print(f"🎯 Test Accuracy: {test_accuracy:.4f}")

# Log test accuracy to wandb
wandb.log({"test_accuracy": test_accuracy})

In [ ]:
# Create confusion matrix
print("📊 Creating confusion matrix...")

cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES)
plt.title('Confusion Matrix - STL-10 Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Save and log to wandb
cm_path = os.path.join(config.RESULTS_PATH, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
wandb.log({"confusion_matrix": wandb.Image(plt)})
plt.show()

print(f"💾 Confusion matrix saved to: {cm_path}")

In [ ]:
# Calculate and plot class-wise accuracy
print("📈 Calculating class-wise accuracy...")

class_accuracies = cm.diagonal() / cm.sum(axis=1)
class_acc_dict = dict(zip(config.CLASS_NAMES, class_accuracies))

# Plot class-wise accuracy
plt.figure(figsize=(12, 6))
bars = plt.bar(config.CLASS_NAMES, class_accuracies, color='skyblue', alpha=0.8, edgecolor='navy')
plt.title('Class-wise Accuracy - STL-10 Classification', fontsize=16, fontweight='bold')
plt.xlabel('Classes', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, class_accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

# Save and log to wandb
acc_path = os.path.join(config.RESULTS_PATH, "class_wise_accuracy.png")
plt.savefig(acc_path, dpi=300, bbox_inches='tight')
wandb.log({"class_wise_accuracy": wandb.Image(plt)})
plt.show()

# Log individual class accuracies to wandb
for class_name, accuracy in class_acc_dict.items():
    wandb.log({f"accuracy_{class_name}": accuracy})

print(f"💾 Class-wise accuracy plot saved to: {acc_path}")

In [ ]:
# Show sample predictions
print("🖼️ Generating sample predictions...")

def show_sample_predictions(model, dataloader, class_names, device, num_correct=10, num_incorrect=10):
    model.eval()
    correct_samples = []
    incorrect_samples = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            if len(correct_samples) >= num_correct and len(incorrect_samples) >= num_incorrect:
                break
                
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            for i in range(images.size(0)):
                if len(correct_samples) >= num_correct and len(incorrect_samples) >= num_incorrect:
                    break
                    
                img = images[i].cpu()
                true_label = labels[i].item()
                pred_label = predicted[i].item()
                
                # Denormalize image for display
                img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                img = torch.clamp(img, 0, 1)
                img = img.permute(1, 2, 0).numpy()
                
                sample_data = {
                    "image": img,
                    "true_label": class_names[true_label],
                    "predicted_label": class_names[pred_label],
                    "correct": true_label == pred_label
                }
                
                if true_label == pred_label and len(correct_samples) < num_correct:
                    correct_samples.append(sample_data)
                elif true_label != pred_label and len(incorrect_samples) < num_incorrect:
                    incorrect_samples.append(sample_data)
    
    return correct_samples, incorrect_samples

# Get sample predictions
correct_samples, incorrect_samples = show_sample_predictions(
    model, test_loader, config.CLASS_NAMES, config.DEVICE, 10, 10
)

print(f"✅ Generated {len(correct_samples)} correct and {len(incorrect_samples)} incorrect samples")

In [ ]:
# Visualize correct predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('✅ Correct Predictions', fontsize=16, fontweight='bold')

for i, sample in enumerate(correct_samples[:10]):
    row = i // 5
    col = i % 5
    
    axes[row, col].imshow(sample['image'])
    axes[row, col].set_title(f"True: {sample['true_label']}\nPred: {sample['predicted_label']}", 
                            fontsize=10, color='green')
    axes[row, col].axis('off')

plt.tight_layout()
wandb.log({"correct_predictions_viz": wandb.Image(plt)})
plt.show()

# Visualize incorrect predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('❌ Incorrect Predictions', fontsize=16, fontweight='bold')

for i, sample in enumerate(incorrect_samples[:10]):
    row = i // 5
    col = i % 5
    
    axes[row, col].imshow(sample['image'])
    axes[row, col].set_title(f"True: {sample['true_label']}\nPred: {sample['predicted_label']}", 
                            fontsize=10, color='red')
    axes[row, col].axis('off')

plt.tight_layout()
wandb.log({"incorrect_predictions_viz": wandb.Image(plt)})
plt.show()

In [ ]:
# Log predictions to wandb as tables
print("📊 Logging predictions to Wandb tables...")

# Create wandb tables
correct_table = wandb.Table(columns=["image", "true_label", "predicted_label", "correct"])
for sample in correct_samples:
    correct_table.add_data(
        wandb.Image(sample["image"], caption=f"True: {sample['true_label']}, Pred: {sample['predicted_label']}"),
        sample["true_label"],
        sample["predicted_label"],
        sample["correct"]
    )

incorrect_table = wandb.Table(columns=["image", "true_label", "predicted_label", "correct"])
for sample in incorrect_samples:
    incorrect_table.add_data(
        wandb.Image(sample["image"], caption=f"True: {sample['true_label']}, Pred: {sample['predicted_label']}"),
        sample["true_label"],
        sample["predicted_label"],
        sample["correct"]
    )

wandb.log({
    "correct_predictions_table": correct_table,
    "incorrect_predictions_table": incorrect_table
})

print("✅ Prediction tables logged to Wandb!")

## 🎓 Exam Sheet Answers

In [ ]:
# Generate final exam answers
print("🎯 FINAL EXAM ANSWERS")
print("="*60)

print(f"\n📊 Question 10 - Test Accuracy: {test_accuracy:.4f}")
print(f"\n📋 Question 11 - Class-wise Accuracy:")
print("-" * 40)

for i, (class_name, accuracy) in enumerate(class_acc_dict.items(), 1):
    print(f"{i:2d}. {class_name:8s}: {accuracy:.4f}")

print("-" * 40)
print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")
print(f"📈 Model: ResNet-18 (pretrained)")
print(f"💾 Model saved to: {best_model_path}")
print(f"🔗 Wandb Dashboard: {wandb.run.url}")

print("\n" + "="*60)
print("✅ STL-10 Classification Complete!")
print("📊 Check your Wandb dashboard for detailed visualizations")
print("📁 Results saved in /content/results/")
print("💾 Model saved in /content/models/")
print("="*60)

# Create summary dictionary for easy access
exam_answers = {
    "test_accuracy": test_accuracy,
    "class_wise_accuracy": class_acc_dict,
    "best_val_accuracy": best_val_acc,
    "wandb_url": wandb.run.url
}

print("\n📋 Exam answers stored in 'exam_answers' variable")

## 🚀 Model Upload Preparation

In [ ]:
# Prepare model for HuggingFace upload
print("📤 Preparing model for HuggingFace upload...")

# Save model in HuggingFace format
hf_model_path = os.path.join(config.MODEL_SAVE_PATH, "pytorch_model.bin")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'num_classes': config.NUM_CLASSES,
        'class_names': config.CLASS_NAMES,
        'test_accuracy': test_accuracy,
        'val_accuracy': best_val_acc
    }
}, hf_model_path)

# Create model card
model_card = f"""---
license: mit
tags:
- image-classification
- stl10
- resnet18
- pytorch
datasets:
- {config.DATASET_NAME}
metrics:
- accuracy: {test_accuracy:.4f}
---

# STL-10 ResNet-18 Classification Model

This model is a fine-tuned ResNet-18 for STL-10 image classification trained on Google Colab.

## Model Details
- **Base Model**: ResNet-18 (pretrained on ImageNet)
- **Dataset**: STL-10 Subset from HuggingFace
- **Classes**: {config.NUM_CLASSES}
- **Test Accuracy**: {test_accuracy:.4f}
- **Validation Accuracy**: {best_val_acc:.4f}
- **Training Platform**: Google Colab

## Class Names
{', '.join(config.CLASS_NAMES)}

## Class-wise Accuracy
"""

for class_name, accuracy in class_acc_dict.items():
    model_card += f"- **{class_name}**: {accuracy:.4f}\n"

model_card += f"""
## Training Configuration
- Batch Size: {config.BATCH_SIZE}
- Learning Rate: {config.LEARNING_RATE}
- Epochs: {config.NUM_EPOCHS}
- Device: {config.DEVICE}

## Usage

```python
import torch
from torchvision import models, transforms
from PIL import Image

# Load model
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 10)

# Load weights (implement loading logic)
# checkpoint = torch.load('pytorch_model.bin')
# model.load_state_dict(checkpoint['model_state_dict'])

# Define transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Inference
image = Image.open("path/to/image.jpg")
input_tensor = transform(image).unsqueeze(0)
with torch.no_grad():
    outputs = model(input_tensor)
    predicted_class = torch.argmax(outputs, dim=1)
```

## Training Results
- Wandb Dashboard: {wandb.run.url}

## Author
Shivam Madhav Kenche (M25CSA028)
"""

# Save model card
readme_path = os.path.join(config.MODEL_SAVE_PATH, "README.md")
with open(readme_path, "w") as f:
    f.write(model_card)

print(f"✅ Model prepared for HuggingFace upload:")
print(f"📄 Model file: {hf_model_path}")
print(f"📋 Model card: {readme_path}")
print(f"\n🔗 To upload to HuggingFace:")
print(f"1. Create repository: kingkenche/stl10-resnet18-colab")
print(f"2. Download files from /content/models/")
print(f"3. Upload to HuggingFace Hub")

# Final wandb log
wandb.log({
    "final_test_accuracy": test_accuracy,
    "final_val_accuracy": best_val_acc,
    "model_ready_for_upload": True
})

print("\n🎉 All tasks completed successfully!")

## 📥 Download Results

In [ ]:
# Create zip file with all results
import zipfile
import glob

def create_results_zip():
    zip_filename = "/content/STL10_Classification_Results.zip"
    
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add model files
        model_files = glob.glob(os.path.join(config.MODEL_SAVE_PATH, "*"))
        for file in model_files:
            arcname = os.path.join("models", os.path.basename(file))
            zipf.write(file, arcname)
            
        # Add result files
        result_files = glob.glob(os.path.join(config.RESULTS_PATH, "*"))
        for file in result_files:
            arcname = os.path.join("results", os.path.basename(file))
            zipf.write(file, arcname)
    
    return zip_filename

# Create summary file
summary_content = f"""# STL-10 Classification Results Summary

## Experiment Details
- Date: {wandb.run.start_time}
- Device: {config.DEVICE}
- Model: ResNet-18
- Dataset: {config.DATASET_NAME}

## Results
- Test Accuracy: {test_accuracy:.4f}
- Best Validation Accuracy: {best_val_acc:.4f}
- Training Epochs: {checkpoint['epoch'] + 1}

## Class-wise Accuracy
"""

for i, (class_name, accuracy) in enumerate(class_acc_dict.items(), 1):
    summary_content += f"{i:2d}. {class_name}: {accuracy:.4f}\n"

summary_content += f"""
## Files Generated
- best_model.pth: Best model checkpoint
- pytorch_model.bin: HuggingFace format model
- README.md: Model card
- confusion_matrix.png: Confusion matrix plot
- class_wise_accuracy.png: Class accuracy bar chart

## Wandb Dashboard
{wandb.run.url}

## GitHub Repository
https://github.com/kingkenche/MLOps-Shivam_Madhav_Kenche-M25CSA028

---
Generated by STL-10 Classification Colab Notebook
Author: Shivam Madhav Kenche (M25CSA028)
"""

# Save summary
summary_path = "/content/EXPERIMENT_SUMMARY.md"
with open(summary_path, "w") as f:
    f.write(summary_content)

# Create zip
zip_path = create_results_zip()

print(f"📦 Results packaged successfully!")
print(f"📥 Download: {zip_path}")
print(f"📄 Summary: {summary_path}")
print(f"\n📋 Files included:")
print(f"   - Best model checkpoint")
print(f"   - HuggingFace model format")
print(f"   - Model card (README)")
print(f"   - Confusion matrix plot")
print(f"   - Class-wise accuracy plot")
print(f"   - Experiment summary")

# Show download link
from IPython.display import HTML
display(HTML(f'<a href="{zip_path}" download>📥 Click here to download results zip file</a>'))
display(HTML(f'<a href="{summary_path}" download>📄 Click here to download experiment summary</a>'))

In [ ]:
# Final cleanup and summary
print("🎉 STL-10 Classification Experiment Complete!")
print("="*60)
print(f"🏆 Final Results:")
print(f"   - Test Accuracy: {test_accuracy:.4f}")
print(f"   - Best Val Accuracy: {best_val_acc:.4f}")
print(f"   - Training Device: {config.DEVICE}")
print(f"   - Total Epochs: {checkpoint['epoch'] + 1}")
print(f"")
print(f"📊 Wandb Dashboard: {wandb.run.url}")
print(f"📁 Results saved in: /content/results/")
print(f"💾 Model saved in: /content/models/")
print(f"📦 Download zip: /content/STL10_Classification_Results.zip")
print("="*60)
print("✅ Ready for GitHub upload and HuggingFace deployment!")

# Finish wandb run
wandb.finish()
print("\n📊 Wandb run completed and saved.")